In [1]:
import cv2
import os
from ultralytics import YOLO

In [10]:
video_dir = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\Running Videos"
folder_dir = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\Running frames"

os.makedirs(folder_dir, exist_ok=True)

for filename in os.listdir(video_dir):
    video_path = os.path.join(video_dir, filename)
    cap = cv2.VideoCapture(video_path)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    for i in range(int(frame_count)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        full_path = os.path.join(folder_dir, filename + "_frame_" + str(i) + ".jpg")
        cv2.imwrite(full_path, frame)
    cap.release()

In [2]:
import json
from ultralytics import YOLO

# --- Config ---
MODEL_PATH = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\best.pt"
FRAMES_DIR = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\Running frames"
OUTPUT_JSON = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\running_frames_predictions_COCO21.json"

BOX_CONF_THRES = 0.25   # min person-detection confidence to keep an annotation
# Mark EVERY predicted keypoint visible (v=2) so all 21 points show up in Roboflow
# for inspection/correction. Set >0.0 if you'd rather hide low-confidence points.
KPT_CONF_THRES = 0.0

# 21-keypoint schema + skeleton, matching the trained model / training categories
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
    "left_big_toe", "right_big_toe", "left_heel", "right_heel",
]

SKELETON = [
    [16, 14], [14, 12], [17, 15], [15, 13], [12, 13], [6, 12], [7, 13],
    [6, 7], [6, 8], [7, 9], [8, 10], [9, 11], [2, 3], [1, 2], [1, 3],
    [2, 4], [3, 5], [4, 6], [5, 7],
    [16, 18], [16, 20], [17, 19], [17, 21],
]

CATEGORIES = [{
    "id": 1,
    "name": "person",
    "supercategory": "person",
    "keypoints": KEYPOINT_NAMES,
    "skeleton": SKELETON,
}]


In [3]:
# Run the trained pose model over every extracted frame and assemble COCO records.
model = YOLO(MODEL_PATH)

images = []
annotations = []
img_id = 0
ann_id = 0

# stream=True yields one Result at a time -> constant memory over thousands of frames
for result in model.predict(source=FRAMES_DIR, stream=True, conf=BOX_CONF_THRES, verbose=False):
    img_id += 1
    h, w = result.orig_shape
    file_name = os.path.basename(result.path)

    images.append({
        "id": img_id,
        "file_name": file_name,
        "width": int(w),
        "height": int(h),
    })

    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        continue  # frame with no detected person: image kept, no annotation

    # .cpu() moves GPU tensors to host memory so .numpy() can read them (no-op on CPU runs)
    # Detections are sorted by confidence (desc) after NMS, so index 0 is the top person.
    # Keep only that single highest-confidence person per frame.
    x1, y1, x2, y2 = result.boxes.xyxy[0].cpu().numpy()
    score = float(result.boxes.conf[0].cpu().numpy())
    kpts = result.keypoints.data[0].cpu().numpy()   # (21, 3) -> x, y, conf (pixels)

    # Clamp corners into the frame so Roboflow import doesn't reject out-of-bounds geometry
    x1 = min(max(float(x1), 0.0), w)
    y1 = min(max(float(y1), 0.0), h)
    x2 = min(max(float(x2), 0.0), w)
    y2 = min(max(float(y2), 0.0), h)

    flat_kpts = []
    num_kpts = 0
    for kx, ky, kconf in kpts:
        v = 2 if kconf >= KPT_CONF_THRES else 0
        if v > 0:
            num_kpts += 1
        # Clamp keypoints into the frame as well
        kx = min(max(float(kx), 0.0), w)
        ky = min(max(float(ky), 0.0), h)
        flat_kpts.extend([round(kx, 2), round(ky, 2), v])

    ann_id += 1
    annotations.append({
        "id": ann_id,
        "image_id": img_id,
        "category_id": 1,
        "iscrowd": 0,
        # COCO bbox is [x_topleft, y_topleft, width, height], derived from the clamped corners
        "bbox": [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
        "area": round(abs((x1 - x2) * (y1 - y2)), 2),
        "keypoints": flat_kpts,
        "num_keypoints": num_kpts,
        "score": round(score, 4),  # detection confidence (prediction metadata)
    })

print(f"Frames processed: {len(images)}")
print(f"Person annotations: {len(annotations)}")


Frames processed: 18842
Person annotations: 18827


In [4]:
# Assemble the single COCO-format document and write it out.
coco = {
    "info": {
        "description": "21-keypoint pose predictions on extracted running frames",
        "source_model": os.path.basename(MODEL_PATH),
    },
    "licenses": [],
    "images": images,
    "annotations": annotations,
    "categories": CATEGORIES,
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(coco, f, indent=4)

print(f"Wrote {len(images)} images and {len(annotations)} annotations to:")
print(OUTPUT_JSON)


Wrote 18842 images and 18827 annotations to:
C:\Users\Abhinav Arora\Desktop\COCO custom model\running_frames_predictions_COCO21.json


Note: Video6 -> annotations: SET VISIBILITY OF UPPER BODY KEYPOINTS TO 0
Same for VIDEO 11 (EVERYTHING EXCEPT HEELS, TOES AND KNEES)
Same for VIDEO16
DOne with 3000 images, starting to finetune best.pt now.